In [ ]:
class DynamicConvBlock(nn.Module):
    def __init__(self, in_channels, reduction_ratio=4):
        super(DynamicConvBlock, self).__init__()
        self.in_channels = in_channels
        self.reduction_ratio = reduction_ratio
        self.reduced_channels = in_channels // reduction_ratio

        # Dynamic weight generation branch
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.fc1 = nn.Conv2d(in_channels, self.reduced_channels, kernel_size=1)
        self.relu = nn.ReLU()
        self.fc2 = nn.Conv2d(self.reduced_channels, in_channels, kernel_size=1)
        self.sigmoid = nn.Sigmoid()

        # Convolutional branch
        self.dynamic_conv = nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1, groups=in_channels)

    def forward(self, x):
        # Generate attention weights
        attention = self.global_pool(x)
        attention = self.fc1(attention)
        attention = self.relu(attention)
        attention = self.fc2(attention)
        attention = self.sigmoid(attention)

        # Apply dynamic convolution
        out = self.dynamic_conv(x) * attention
        return out